In [1]:
import torch
import numpy as np
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

Running on: cuda


## The Components
Before the loop, we need two things:
<ol>
<li>The Loss Function: How wrong is the model?</li>

nn.MSELoss(): For regression.

nn.CrossEntropyLoss(): For classification (Combines Softmax + LogLikelihood).

<li>The Optimizer: How do we update weights?</li>

torch.optim.SGD, torch.optim.Adam, etc.

Crucial: You must tell the optimizer what to optimize.

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)</ol>

## The 5-Step Ritual
Inside every training step (batch), you perform a specific ritual. Memorize this sequence.
<ol>
<li>optimizer.zero_grad(): Clear old gradients. (If you forget this, PyTorch adds the new gradients to the old ones, and your training explodes).</li>

<li>output = model(input): The Forward Pass.</li>

<li>loss = criterion(output, target): Calculate error.</li>

<li>loss.backward(): The Backward Pass (Compute gradients).</li>

<li>optimizer.step(): Update weights using the gradients.</li><ol>

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# --- Setup (Review) ---
# 1. Data
X = torch.randn(100, 20)
Y = torch.randint(0, 2, (100,)) # Binary classification labels
dataset = TensorDataset(X, Y)   # Quick way to wrap tensors into a Dataset
loader = DataLoader(dataset, batch_size=10, shuffle=True)

# 2. Model
model = nn.Sequential(
    nn.Linear(20, 64),
    nn.ReLU(),
    nn.Linear(64, 2) # Output size 2 for CrossEntropy
)

# --- NEW: Optimizer & Loss ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# --- The Training Loop ---
EPOCHS = 5

for epoch in range(EPOCHS):
    
    # Track metrics manually
    total_loss = 0
    
    # Switch model to training mode (Important for Dropout/BatchNorm)
    model.train()
    
    for batch_idx, (data, target) in enumerate(loader):
        
        # Step 1: Zero Gradients
        optimizer.zero_grad()
        
        # Step 2: Forward Pass
        predictions = model(data)
        
        # Step 3: Compute Loss
        loss = criterion(predictions, target)
        
        # Step 4: Backward Pass
        loss.backward()
        
        # Step 5: Update Weights
        optimizer.step()
        
        # Accumulate loss for printing (use .item() to get float)
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1} | Avg Loss: {total_loss / len(loader):.4f}")

print("Training Complete.")

Epoch 1 | Avg Loss: 0.7350
Epoch 2 | Avg Loss: 0.6086
Epoch 3 | Avg Loss: 0.5038
Epoch 4 | Avg Loss: 0.4309
Epoch 5 | Avg Loss: 0.3645
Training Complete.


### Deep Dive for Researchers: model.train() vs model.eval()
You will see these toggles often. They change how specific layers behave.

model.train(): Tells the model "we are learning".

Dropout: Randomly zeros out neurons.

BatchNorm: Updates running statistics (mean/var) based on the current batch.

model.eval(): Tells the model "we are testing/predicting".

Dropout: Disabled (all neurons active).

BatchNorm: Uses the frozen running statistics (no updates).

Common Bug: Training a model, getting great accuracy, then running inference and getting garbage results... because you forgot to call model.eval().

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# --- Step 1: Define the Device ---
# Standard boilerplate to pick the best available hardware
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Training on device: {device}")

# --- Data Preparation ---
X = torch.randn(1000, 10)
Y = torch.randint(0, 3, (1000,))

dataset = TensorDataset(X, Y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# --- Model Definition ---
class SimpleClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(SimpleClassifier, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.layer2 = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

# Instantiate
model = SimpleClassifier(input_size=10, hidden_size=32, num_classes=3)

# --- Step 2: Move Model to Device ---
# Crucial! This moves all weights/biases to the GPU VRAM.
# Doing this BEFORE creating the optimizer is best practice.
model = model.to(device)

# --- Setup Optimizer & Loss ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# --- The Training Loop ---
EPOCHS = 5000

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    for i, (inputs, labels) in enumerate(loader):
        # --- Step 3: Move Batch Data to Device ---
        # We move data only when we need it to save VRAM.
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        # A. Zero Gradients
        optimizer.zero_grad()
        
        # B. Forward Pass 
        # (Everything matches now: Model is on GPU, Inputs are on GPU)
        outputs = model(inputs)
        
        # C. Calculate Loss
        loss = criterion(outputs, labels)
        
        # D. Backward Pass
        loss.backward()
        
        # E. Update Weights
        optimizer.step()
        
        running_loss += loss.item()
    
    avg_loss = running_loss / len(loader)
    
    if (epoch + 1) % 200 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}")

print("Training Complete.")

Training on device: cuda
Epoch [200/5000], Loss: 1.0355
Epoch [400/5000], Loss: 0.9889
Epoch [600/5000], Loss: 0.9718
Epoch [800/5000], Loss: 0.9435
Epoch [1000/5000], Loss: 0.9133
Epoch [1200/5000], Loss: 0.9050
Epoch [1400/5000], Loss: 0.8712
Epoch [1600/5000], Loss: 0.8431
Epoch [1800/5000], Loss: 0.8188
Epoch [2000/5000], Loss: 0.8007
Epoch [2200/5000], Loss: 0.7970
Epoch [2400/5000], Loss: 0.7840
Epoch [2600/5000], Loss: 0.7814
Epoch [2800/5000], Loss: 0.7901
Epoch [3000/5000], Loss: 0.7695
Epoch [3200/5000], Loss: 0.7663
Epoch [3400/5000], Loss: 0.7603
Epoch [3600/5000], Loss: 0.7641
Epoch [3800/5000], Loss: 0.7537
Epoch [4000/5000], Loss: 0.7566
Epoch [4200/5000], Loss: 0.7410
Epoch [4400/5000], Loss: 0.7524
Epoch [4600/5000], Loss: 0.7712
Epoch [4800/5000], Loss: 0.7562
Epoch [5000/5000], Loss: 0.7499
Training Complete.
